# Fuego contra Fuego


Instalamos, cargamos y seteamos el entorno

In [10]:
%pip install optuna


Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from sklearn.tree import DecisionTreeClassifier, plot_tree,  _tree
from sklearn.model_selection import train_test_split
from sklearn.model_selection import ShuffleSplit, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

from joblib import Parallel, delayed

import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_slice, plot_contour

from time import time

import pickle


In [12]:
from pathlib import Path

project_path = Path('/Users/thenseler/Documents/Maestria/DMEyF/dmeyf2026')
dataset_dir = project_path / 'data'
modelos_dir = project_path / 'modelos'
modelos_dir.mkdir(parents=True, exist_ok=True)
dataset_path = dataset_dir.as_posix() + '/'
modelos_path = modelos_dir.as_posix() + '/'
db_path = dataset_path
dataset_file = 'competencia_01.csv'
dataset_file_path = dataset_dir / dataset_file
if not dataset_file_path.is_file():
    raise FileNotFoundError(f'No se encontró el dataset local: {dataset_file_path}')
print(f'Cargando dataset local: {dataset_file_path}')

ganancia_acierto = 1_072_500
costo_estimulo = 27_500

mes_train = 202103
mes_test = 202105

# agregue sus semillas
semilla = 17
semillas = np.random.choice(1000000, size=50, replace=False).tolist()

data = pd.read_csv(dataset_file_path)

Cargando dataset local: /Users/thenseler/Documents/Maestria/DMEyF/dmeyf2026/data/competencia_01.csv


Seguimos trabajando con Marzo como entrenamiento y Mayo como test.

In [13]:
X = data[data['foto_mes'] == mes_train]
y = X['clase_ternaria']
X = X.drop(columns=['clase_ternaria'])

In [14]:
X_futuro = data[data['foto_mes'] == mes_test]
y_futuro = X_futuro['clase_ternaria']
X_futuro = X_futuro.drop(columns=['clase_ternaria'])

Y variaremos la forma de la función de ganancia, para poder ser utilizada de una forma más genérica.

In [16]:
def ganancia_prob(y_hat, y, prop=1, class_index=1, threshold=0.025):
  @np.vectorize
  def ganancia_row(predicted, actual, threshold=0.025):
    return  (predicted >= threshold) * (ganancia_acierto if actual == "BAJA+2" else -costo_estimulo)

  return ganancia_row(y_hat[:,class_index], y).sum() / prop



Entrenemos y recordemos cuanto nos daba la ganancia en **Mayo**

Hablemos de ensemables, qué son?

* Un ensemble de modelos es una técnica donde se combinan múltiples modelos individuales para mejorar la precisión y robustez de las predicciones. La idea es que al combinar varios modelos, se pueden aprovechar las fortalezas de cada uno y reducir la posibilidad de errores que podría cometer un único modelo.

**Tipos de ensemble**:
 * **Bagging (Bootstrap Aggregating)**: Consiste en entrenar varios modelos base en diferentes subconjuntos del conjunto de datos de entrenamiento obtenidos mediante técnicas de remuestreo como el bootstrap y luego promediar sus predicciones. Ejemplo: **Random Forest**
 * **Boosting**: En esta técnica, los modelos se entrenan de manera secuencial. Cada modelo intenta corregir los errores cometidos por el modelo anterior. Ejemplo: AdaBoost y **Gradient Boosting**.
 * **Stacking**: En el stacking, se entrenan varios modelos y se combinan usando un "modelo meta". Las predicciones de los modelos base sirven como features para entrenar este modelo meta, que produce la predicción final.

* **Ventajas de usar ensemble de modelos**:
 * **Mejor rendimiento**: Al combinar modelos, generalmente se mejora la precisión en comparación con un solo modelo.
 * **Robustez**: Al integrar diferentes modelos, se mitiga el riesgo de que los errores de un modelo individual afecten gravemente la predicción final.




Pongamos foco en el **Random Forest**

Es un algoritmo de aprendizaje automático que funciona creando un conjunto de árboles de decisión. Para la creación de **árboles distintos** utiliza una técnica llamada bagging para crear múltiples subconjuntos del conjunto de datos de entrenamiento. Cada subconjunto se genera seleccionando al azar muestras del conjunto de datos original con reemplazo. No usa la totalidad de los datos de entrenamiento para cada conjunto. Los datos que quedan fueran son conocidos como **Out of Bag (oob)**

Para cada subconjunto, se construye un árbol de decisión. Sin embargo en cada nodo del árbol, **Random Forest** selecciona de forma aleatoria un grupo de variables y ajusta el árbol con esas variables. Este proceso ayuda a crear árboles que son menos correlacionados entre sí.

Cada árbol en el bosque se entrena de manera independiente usando su respectivo subconjunto de datos. Lueago, para una nueva observación, cada árbol realiza una predicción. El Random Forest luego combina las predicciones de todos los árboles para hacer una predicción final, devolviendo el promedio de las probabilidades de cada árbol individual.



Como desde la clase pasada solo personas más inteligentes, no vamos a empezar a probar **Random Forest** simples. Vamos a parametrizarlo desde el vamos.

Primero vamos a entender algunas limitaciones de la implementación:

El **Random Forest** no soporta nulos! Shame on you sklearn!.

Vamos a tener que imputar los datos. Discutamos entre todos forma de imputar los datos, mientras para salir del paso usamos la peor de todas.

In [17]:
imp_median = SimpleImputer(missing_values=np.nan, strategy='median')
Xi = imp_median.fit_transform(X)
Xif = imp_median.fit_transform(X_futuro)

Los parámetros que se pueden ajustar en el **rf** son

1. **n_estimators**: Número de árboles en el bosque.
2. **max_depth**: Profundidad máxima de los árboles.
3. **min_samples_split**: Número mínimo de muestras requeridas para dividir un nodo interno.
4. **min_samples_leaf**: Número mínimo de muestras requeridas para estar en un nodo hoja.
5. **max_features**: Número de features a usar en cada árbol. **sqrt** es una elección histórica.
6. **max_leaf_nodes**: Número máximo de nodos hoja en cada árbol.
7. **oob_score**: Indica si se usa la muestra fuera de bolsa (out-of-bag) para estimar la calidad del modelo. Para evitar hacer un **montecarlo-cross-validation** que se toma su tiempo, usaremos esta opción para buscar el mejor modelo. No es la mejor opción. Pero no es tan mala.
8. **n_jobs**: Siempre -1, para que use todos los cores presentes en 9. **max_samples**: Fracción de los samples.

Finalmente nuestra función de optimización queda la siguiente forma:

In [18]:
def objective(trial):
    max_depth = trial.suggest_int('max_depth', 2, 32)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 2000)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 200)
    max_features = trial.suggest_float('max_features', 0.05, 0.7)

    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

    model.fit(Xi, y)

    return ganancia_prob(model.oob_decision_function_, y)

storage_name = "sqlite:///" + db_path + "optimization_rf.db"
study_name = "exp_401_random-forest-opt"

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_name,
    load_if_exists=True,
)



[I 2026-08-31 19:40:31,778] A new study created in RDB with name: exp_401_random-forest-opt


In [19]:
study.optimize(objective, n_trials=100)

[I 2026-08-31 19:41:29,106] Trial 0 finished with value: 326205000.0 and parameters: {'max_depth': 29, 'min_samples_split': 1406, 'min_samples_leaf': 106, 'max_features': 0.6547101311340564}. Best is trial 0 with value: 326205000.0.
[I 2026-08-31 19:41:45,740] Trial 1 finished with value: 288392500.0 and parameters: {'max_depth': 5, 'min_samples_split': 1912, 'min_samples_leaf': 133, 'max_features': 0.4888266003530553}. Best is trial 0 with value: 326205000.0.
[I 2026-08-31 19:41:55,893] Trial 2 finished with value: 301042500.0 and parameters: {'max_depth': 9, 'min_samples_split': 965, 'min_samples_leaf': 152, 'max_features': 0.15458951824213432}. Best is trial 0 with value: 326205000.0.
[I 2026-08-31 19:42:20,099] Trial 3 finished with value: 323785000.0 and parameters: {'max_depth': 25, 'min_samples_split': 1235, 'min_samples_leaf': 82, 'max_features': 0.2954643379795959}. Best is trial 0 with value: 326205000.0.
[I 2026-08-31 19:42:33,698] Trial 4 finished with value: 259985000.0 an

Exploramos como fue la búsqueda de parámetros

In [20]:
optuna.visualization.plot_optimization_history(study)

In [21]:
plot_param_importances(study)

In [22]:
plot_slice(study)

In [23]:
plot_contour(study)

In [24]:
plot_contour(study, params=["max_depth", "min_samples_split"])

Ajustamos el mejor modelo a todo el conjunto de datos



In [25]:
model_rf = RandomForestClassifier(
        n_estimators=100,
        **study.best_params,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

model_rf.fit(Xi, y)


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",18
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",482
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",152
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.4267960026837606
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",19828
,"max_samples max_samples: int or float, default=NoneIf bootstrap is True, the number of samples to draw from Xto train each base estimator.- If None (default), then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples... versionadded:: 0.22.. versionchanged:: 1.9 Float `max_samples` is relative to `sample_weight.sum()` instead of `X.shape[0]` for weighted samples.",0.7
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported 

Guardamos el modelo, para no tener que optimizar cada vez que lo queramos usar.

In [26]:
filename = modelos_path + 'exp_206_random_forest_model_100.sav'
pickle.dump(model_rf, open(filename, 'wb'))


Y lo cargamos para utilizarlo

In [27]:
model_rf = pickle.load(open(filename, 'rb'))
model_rf

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",18
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",482
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",152
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.4267960026837606
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",19828
,"max_samples max_samples: int or float, default=NoneIf bootstrap is True, the number of samples to draw from Xto train each base estimator.- If None (default), then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples... versionadded:: 0.22.. versionchanged:: 1.9 Float `max_samples` is relative to `sample_weight.sum()` instead of `X.shape[0]` for weighted samples.",0.7
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported 

Ahora vamos medir su ganancia sobre el dataset de **Mayo**

In [28]:
y_pred_rf = model_rf.predict_proba(Xif)
ganancias_rf = ganancia_prob(y_pred_rf, y_futuro)
print(f"Ganancia de modelo RF: {ganancias_rf}")

Ganancia de modelo RF: 240790000.0


Frente a los 208.010.000 de mi mejor árbol es algo sustancialmente mejor.

Igualmente usamos poquitos estimadores. Tan solo unos 100. Suficientes como para que la optimización no tarde una eternidad, pero es muy poco. Deberíamos sumar unos cuantos más.

In [29]:
model_rf_1000 = RandomForestClassifier(
        n_estimators=1000,
        **study.best_params,
        max_samples=0.7,
        random_state=semillas[0],
        n_jobs=-1,
        oob_score=True
    )

model_rf_1000.fit(Xi, y)

filename_rf_1000 = modelos_path + 'exp_206_random_forest_model_1000.sav'
pickle.dump(model_rf, open(filename_rf_1000, 'wb'))


In [30]:
model_rf_1000 = pickle.load(open(filename_rf_1000, 'rb'))
model_rf_1000

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",18
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",482
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",152
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.4267960026837606
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",19828
,"max_samples max_samples: int or float, default=NoneIf bootstrap is True, the number of samples to draw from Xto train each base estimator.- If None (default), then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples... versionadded:: 0.22.. versionchanged:: 1.9 Float `max_samples` is relative to `sample_weight.sum()` instead of `X.shape[0]` for weighted samples.",0.7
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported 

Veamos si sumar 10 veces más estimadores tuvo algún efecto

In [31]:
y_pred_rf = model_rf_1000.predict_proba(Xif)
ganancias_rf = ganancia_prob(y_pred_rf, y_futuro)
print(f"Ganancia de modelo RF 1000: {ganancias_rf}")

Ganancia de modelo RF 1000: 240790000.0


Mejoró un poco! Cada peso vale.



Por último evaluamos cuales son las variables más importantes del modelo. ¿Habrá influido la imputación?

In [32]:
importances = model_rf.feature_importances_

features = X.columns
feat_importances = pd.DataFrame({'feature': features, 'importance': importances})
feat_importances = feat_importances.sort_values('importance', ascending=False)

feat_importances.head(25)


,feature,importance
107,ctrx_quarter,0.223544
22,mcuentas_saldo,0.084224
18,mcaja_ahorro,0.079614
11,mpasivos_margen,0.069399
33,mprestamos_personales,0.065046
16,mcuenta_corriente,0.046558
28,mtarjeta_visa_consumo,0.032198
2,active_quarter,0.031967
8,mrentabilidad_annual,0.031655
32,cprestamos_personales,0.027282


Vamos a hacer un par de experimentos, que pasa si sacamos la variable más importante?

In [33]:
most_important_feature = feat_importances.iloc[0]['feature']

X_df = pd.DataFrame(Xi, columns=X.columns)
X_futuro_df = pd.DataFrame(Xif, columns=X_futuro.columns)

X_no_best_feature = X_df.drop(columns=[most_important_feature])
X_futuro_no_best_feature = X_futuro_df.drop(columns=[most_important_feature])

model_rf_no_best_feature = RandomForestClassifier(
    n_estimators=100,
    **study.best_params,
    max_samples=0.7,
    random_state=semillas[0],
    n_jobs=-1,
    oob_score=True
)

print(f"Entrenando modelo RF sin '{most_important_feature}'...")
model_rf_no_best_feature.fit(X_no_best_feature, y)

y_pred_rf_no_best_feature = model_rf_no_best_feature.predict_proba(X_futuro_no_best_feature)

ganancia_rf_no_best_feature = ganancia_prob(y_pred_rf_no_best_feature, y_futuro)

print(f"Ganancia del modelo RF sin '{most_important_feature}': {ganancia_rf_no_best_feature}")
print(f"Ganancia del modelo RF original: {ganancias_rf}")

Entrenando modelo RF sin 'ctrx_quarter'...
Ganancia del modelo RF sin 'ctrx_quarter': 251570000.0
Ganancia del modelo RF original: 240790000.0


¿Qué paso? ¿Se ha derrumbado las ganancias frente a la ausencia del pilar de los árboles?

Una prueba más, que pasa si sumamos variables que son ruido puro.

In [35]:
X_df_from_Xi = pd.DataFrame(Xi, columns=X.columns)

np.random.seed(semilla)

random_features_train = pd.DataFrame({
    f'random_feature_{i}': np.random.rand(len(X_df_from_Xi)) for i in range(1, 6)
})

X_with_random_features = pd.concat([X_df_from_Xi, random_features_train], axis=1)

model_rf_new_features = RandomForestClassifier(
    n_estimators=100,
    **study.best_params,
    max_samples=0.7,
    random_state=semillas[0],
    n_jobs=-1,
    oob_score=True
)

model_rf_new_features.fit(X_with_random_features, y)

importances_new_features = model_rf_new_features.feature_importances_

features_new_features = X_with_random_features.columns
feat_importances_new_features = pd.DataFrame({'feature': features_new_features, 'importance': importances_new_features})
feat_importances_new_features = feat_importances_new_features.sort_values('importance', ascending=False)

display(feat_importances_new_features.head(25))

,feature,importance
107,ctrx_quarter,0.222830
18,mcaja_ahorro,0.089438
22,mcuentas_saldo,0.086300
11,mpasivos_margen,0.073710
33,mprestamos_personales,0.060474
16,mcuenta_corriente,0.040765
32,cprestamos_personales,0.034461
28,mtarjeta_visa_consumo,0.031662
8,mrentabilidad_annual,0.030532
29,ctarjeta_master,0.024558


Y una más, cual es el área bajo la curva en train?

In [36]:
from sklearn.metrics import roc_auc_score

y_pred_rf_train = model_rf_1000.predict_proba(Xi)

ganancia_rf_train = ganancia_prob(y_pred_rf_train, y)
print(f"Ganancia en el conjunto de entrenamiento: {ganancia_rf_train}")
print(f"Ganancia en el conjunto de Mayo: {ganancias_rf}")

y_train_numeric = (y == 'BAJA+2').astype(int)
y_futuro_numeric = (y_futuro == 'BAJA+2').astype(int)

auc_train = roc_auc_score(y_train_numeric, y_pred_rf_train[:, 1])
print(f"Área bajo la curva (AUC) en el conjunto de entrenamiento: {auc_train}")

auc_mayo = roc_auc_score(y_futuro_numeric, y_pred_rf[:, 1])
print(f"Área bajo la curva (AUC) en el conjunto de Mayo: {auc_mayo}")

Ganancia en el conjunto de entrenamiento: 425590000.0
Ganancia en el conjunto de Mayo: 240790000.0
Área bajo la curva (AUC) en el conjunto de entrenamiento: 0.9420289793544935
Área bajo la curva (AUC) en el conjunto de Mayo: 0.8839934848166041


Cuanta diferencia entre conjuntos. Seguramente debemos descartar este algoritmo, ¿o no?

Pregunta:
* **¿Cómo sabe que un random forest es superior a otro?**
* **¿Cómo supera el random forest presentado?**

## Tarea:

* Mejore la parametrización del **rf**
